In [4]:
pp = build_pairs(pairs, table)
pp.to_csv(out_dir / "pairs.csv", index=False)

chk = pd.read_csv(out_dir / "pairs.csv")
nt = chk[chk["near_tie"] == 1]
fl = nt[nt["flipped"] == 1]
print(chk.shape, len(nt))
print(round(nt["front_ahead"].mean(), 4))
print(len(fl), round(fl["slot_ahead_won"].mean(), 4))

(4861, 15) 1943
0.5347
108 0.6019


In [3]:
q = pd.read_sql(Path("../sql/04_qualifying_times.sql").read_text(), con)

q["group"] = np.where(q["q3_millis"].notna(), 3,
                      np.where(q["q2_millis"].notna(), 2, 1))
q["t"] = q["q3_millis"].fillna(q["q2_millis"]).fillna(q["q1_millis"])

q = q.sort_values(["race_id", "qpos"])
back = q.groupby("race_id")[["qpos", "group", "t", "driver_id"]].shift(-1)
pairs = q[["race_id", "year", "driver_id", "qpos", "group", "t"]].join(
    back.add_suffix("_back")
)

ok = (pairs["group"] == pairs["group_back"]) & (pairs["qpos_back"] == pairs["qpos"] + 1)
pairs = pairs[ok].copy()
pairs["gap_ms"] = pairs["t_back"] - pairs["t"]
print(pairs["race_id"].nunique(), len(pairs))

393 6966


In [2]:
raw = pd.read_sql(Path("../sql/01_race_results.sql").read_text(), con)
chron = pd.read_sql("SELECT * FROM constructor_chronology", con)
table = build_table(raw, chron)
print(table.shape)

(8636, 21)


In [1]:
import sys
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../src").resolve()))

from clean import build_table
from export import build_pairs

con = sqlite3.connect("../data/raw/f1db.db")
out_dir = Path("../data/processed")